Call the API to the dataset and fetch the data from the API

In [1]:
import requests
import pandas as pd
import io

dataset_id = "d_0f2f47515425404e6c9d2a040dd87354"
url = "https://api-open.data.gov.sg/v1/public/api/datasets/" + dataset_id + "/poll-download"

response = requests.get(url)
json_data = response.json()
if json_data['code'] != 0:
    print(json_data['errMsg'])
    exit(1)

download_url = json_data['data']['url']
response = requests.get(download_url)

# Try CSV first (common for data.gov.sg), fallback to JSON
try:
    df = pd.read_csv(io.StringIO(response.text))
except (ValueError, pd.errors.ParserError):
    data = response.json()
    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif "records" in data:
        df = pd.DataFrame(data["records"])
    elif "result" in data and "records" in data["result"]:
        df = pd.DataFrame(data["result"]["records"])
    else:
        df = pd.json_normalize(data)

# Flatten GeoJSON FeatureCollection if present
if "features" in df.columns and len(df) > 0:
    first = df.iloc[0]
    if first.get("type") == "FeatureCollection" and isinstance(first.get("features"), list):
        rows = []
        for feat in first["features"]:
            row = dict(feat.get("properties", {}))
            geom = feat.get("geometry", {})
            if geom.get("type") == "Point" and "coordinates" in geom:
                row["longitude"] = geom["coordinates"][0]
                row["latitude"] = geom["coordinates"][1]
            rows.append(row)
        df = pd.DataFrame(rows)

df

,OBJECTID_1,URL_PATH,IMAGE_PATH,IMAGE_ALT_TEXT,PHOTOCREDITS,PAGETITLE,LASTMODIFIED,LATITUDE,LONGTITUDE,ADDRESS,POSTALCODE,OVERVIEW,EXTERNAL_LINK,META_DESCRIPTION,OPENING_HOURS,INC_CRC,FMEL_UPD_D,longitude,latitude
0,1001,www.yoursingapore.com/en/see-do-singapore/arts...,www.yoursingapore.com/content/dam/desktop/glob...,NaN,Â©Darren Soh/National Gallery,National Gallery Singapore,2015-10-23T18:03:40.939+08:00,1.290,103.851356,1 St Andrew's Road,NaN,Take in the regionâ€™s newest and largest muse...,http://www.nationalgallery.sg,Take in the regionâ€™s newest and largest muse...,"Effective from 24 November 2015, Sunday to Thu...",8A41138FE0684CA7,20180619175243,103.851356,1.290
1,1002,www.yoursingapore.com/en/see-do-singapore/cult...,NaN,NaN,NaN,Sultan Mosque (Masjid Sultan) Singapore,2015-11-02T10:27:45.190+08:00,1.302,103.859171,3 Muscat Street,NaN,"Also known as Masjid Sultan, the impressive Su...",http://www.sultanmosque.org.sg,The impressive Sultan Mosque (also known as Ma...,Monday to Sunday:9.30am â€“ 12pm and 2pm â€“ 4...,A3218DB77F17BFC6,20180619175243,103.859171,1.302
2,1003,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,Sri Mariamman Temple's ornate and elaborate de...,Joel Chua DY,Sri Mariamman Temple: Hindu Temple in Singapore,2015-11-02T10:31:58.848+08:00,1.282,103.845380,244 South Bridge Road,NaN,"Located in Chinatown, the Sri Mariamman Temple...",http://heb.gov.sg/our-subsidiaries/temples/sri...,"Sri Mariamman Temple, located in Chinatown, is...","Daily from 7am â€“ 12pm, and 6pm â€“ 9pm",F66A0BB9C79777F1,20180619175243,103.845380,1.282
3,1004,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,The Armenian Church in Singapore is a 19th-cen...,Joel Chua DY,Armenian Church in Singapore,2015-11-02T10:34:55.710+08:00,1.293,103.849660,60 Hill Street,NaN,The oldest Christian church in Singapore is an...,http://www.armeniansinasia.org/,The Armenian Church is the oldest Christian ch...,"Daily, 9am â€“6pm",4C9C762C0E30783B,20180619175243,103.849660,1.293
4,1005,www.yoursingapore.com/en/see-do-singapore/arch...,www.yoursingapore.com/content/dam/desktop/glob...,"The charm of CHIJMES, Singapore harks back to ...",NaN,CHIJMES Singapore,2015-11-02T10:13:35.220+08:00,1.295,103.851680,30 Victoria Street,NaN,Whether functioning as a school or a lifestyle...,http://chijmes.com.sg/,Who would have thought a former school would m...,"Daily, 24 hours,Opening hours of businesses in...",5A0FF621153ABB8F,20180619175243,103.851680,1.295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,996,www.yoursingapore.com/en/see-do-singapore/natu...,www.yoursingapore.com/content/dam/desktop/glob...,MacRitchie Reservoir is natureâ€™s playground ...,NaN,MacRitchie Singapore & Singapore Nature Reserve,2015-10-15T18:52:17.740+08:00,1.345,103.822346,MacRitchie Reservoir Park,NaN,The attractions around MacRitchie Reservoir is...,http://www.nparks.gov.sg/cms/macritchie.php?op...,The attractions around MacRitchie Reservoir ca...,NaN,55CEE55B4F6A895A,20180619175243,103.822346,1.345
105,997,www.yoursingapore.com/en/see-do-singapore/natu...,www.yoursingapore.com/content/dam/desktop/glob...,Be mesmerised by stunning waterfront views at ...,Derrick See,Gardens by the Bay,2015-10-19T15:26:06.746+08:00,1.282,103.863613,18 Marina Gardens Drive,NaN,This sprawling garden in the city provides mes...,http://www.gardensbythebay.com.sg/en/home.html,Gardens by the Bay provides mesmerising waterf...,"Bay South Outdoor Gardens Daily, 5am - 2am Coo...",76915C78CEAAEB32,20180619175243,103.863613,1.282
106,998,www.yoursingapore.com/en/see-do-singapore/plac...,www.yoursingapore.com/content/dam/desktop/glob...,Immerse yourself in a day of history and cultu...,NaN,"Civic District, Singapore",2015-10-13T14:04:28.278+08:00,1.293,103.852220,150 North Bridge Road,NaN,The Civic District is where Singaporeâ€™s hist...,NaN,The Civic District is where Sing

Flattening tourist attractions from a **local GeoJSON** (`TouristAttractions.geojson`). That file is **not** in this repo—save/export it next to the notebook or repo root if you use this path. **If you already ran the poll-download cell above**, it left a flattened `df` in memory; the next cell will reuse that when the file is missing.

In [4]:
import json
from pathlib import Path

import pandas as pd

# Relative paths use the Jupyter kernel cwd (often repo root), not the notebook folder.
_candidates = [
    Path.cwd() / "TouristAttractions.geojson",
    Path.cwd() / "tourist_attraction_ingest" / "TouristAttractions.geojson",
]
_geo_path = next((p for p in _candidates if p.is_file()), None)

if _geo_path is not None:
    with _geo_path.open(encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    for feat in data.get("features", []):
        row = dict(feat.get("properties", {}))
        geom = feat.get("geometry", {}) or {}
        if geom.get("type") == "Point" and "coordinates" in geom:
            row["longitude"] = geom["coordinates"][0]
            row["latitude"] = geom["coordinates"][1]
        rows.append(row)
    df = pd.DataFrame(rows)
elif "df" in globals() and isinstance(df, pd.DataFrame) and len(df) > 0:
    print(
        "TouristAttractions.geojson not found — using existing `df` from the API cell above.\n"
        "Searched:\n  " + "\n  ".join(str(p) for p in _candidates)
    )
else:
    raise FileNotFoundError(
        "TouristAttractions.geojson not found. Searched:\n  "
        + "\n  ".join(str(p) for p in _candidates)
        + "\n\nEither save the GeoJSON to one of those paths, or run the previous cell "
        "that loads data from data.gov.sg (poll-download) so `df` exists."
    )

df

TouristAttractions.geojson not found — using existing `df` from the API cell above.
Searched:
  /root/hdb-price-estimator/tourist_attraction_ingest/TouristAttractions.geojson
  /root/hdb-price-estimator/tourist_attraction_ingest/tourist_attraction_ingest/TouristAttractions.geojson


,OBJECTID_1,URL_PATH,IMAGE_PATH,IMAGE_ALT_TEXT,PHOTOCREDITS,PAGETITLE,LASTMODIFIED,LATITUDE,LONGTITUDE,ADDRESS,POSTALCODE,OVERVIEW,EXTERNAL_LINK,META_DESCRIPTION,OPENING_HOURS,INC_CRC,FMEL_UPD_D,longitude,latitude
0,1001,www.yoursingapore.com/en/see-do-singapore/arts...,www.yoursingapore.com/content/dam/desktop/glob...,NaN,Â©Darren Soh/National Gallery,National Gallery Singapore,2015-10-23T18:03:40.939+08:00,1.290,103.851356,1 St Andrew's Road,NaN,Take in the regionâ€™s newest and largest muse...,http://www.nationalgallery.sg,Take in the regionâ€™s newest and largest muse...,"Effective from 24 November 2015, Sunday to Thu...",8A41138FE0684CA7,20180619175243,103.851356,1.290
1,1002,www.yoursingapore.com/en/see-do-singapore/cult...,NaN,NaN,NaN,Sultan Mosque (Masjid Sultan) Singapore,2015-11-02T10:27:45.190+08:00,1.302,103.859171,3 Muscat Street,NaN,"Also known as Masjid Sultan, the impressive Su...",http://www.sultanmosque.org.sg,The impressive Sultan Mosque (also known as Ma...,Monday to Sunday:9.30am â€“ 12pm and 2pm â€“ 4...,A3218DB77F17BFC6,20180619175243,103.859171,1.302
2,1003,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,Sri Mariamman Temple's ornate and elaborate de...,Joel Chua DY,Sri Mariamman Temple: Hindu Temple in Singapore,2015-11-02T10:31:58.848+08:00,1.282,103.845380,244 South Bridge Road,NaN,"Located in Chinatown, the Sri Mariamman Temple...",http://heb.gov.sg/our-subsidiaries/temples/sri...,"Sri Mariamman Temple, located in Chinatown, is...","Daily from 7am â€“ 12pm, and 6pm â€“ 9pm",F66A0BB9C79777F1,20180619175243,103.845380,1.282
3,1004,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,The Armenian Church in Singapore is a 19th-cen...,Joel Chua DY,Armenian Church in Singapore,2015-11-02T10:34:55.710+08:00,1.293,103.849660,60 Hill Street,NaN,The oldest Christian church in Singapore is an...,http://www.armeniansinasia.org/,The Armenian Church is the oldest Christian ch...,"Daily, 9am â€“6pm",4C9C762C0E30783B,20180619175243,103.849660,1.293
4,1005,www.yoursingapore.com/en/see-do-singapore/arch...,www.yoursingapore.com/content/dam/desktop/glob...,"The charm of CHIJMES, Singapore harks back to ...",NaN,CHIJMES Singapore,2015-11-02T10:13:35.220+08:00,1.295,103.851680,30 Victoria Street,NaN,Whether functioning as a school or a lifestyle...,http://chijmes.com.sg/,Who would have thought a former school would m...,"Daily, 24 hours,Opening hours of businesses in...",5A0FF621153ABB8F,20180619175243,103.851680,1.295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,996,www.yoursingapore.com/en/see-do-singapore/natu...,www.yoursingapore.com/content/dam/desktop/glob...,MacRitchie Reservoir is natureâ€™s playground ...,NaN,MacRitchie Singapore & Singapore Nature Reserve,2015-10-15T18:52:17.740+08:00,1.345,103.822346,MacRitchie Reservoir Park,NaN,The attractions around MacRitchie Reservoir is...,http://www.nparks.gov.sg/cms/macritchie.php?op...,The attractions around MacRitchie Reservoir ca...,NaN,55CEE55B4F6A895A,20180619175243,103.822346,1.345
105,997,www.yoursingapore.com/en/see-do-singapore/natu...,www.yoursingapore.com/content/dam/desktop/glob...,Be mesmerised by stunning waterfront views at ...,Derrick See,Gardens by the Bay,2015-10-19T15:26:06.746+08:00,1.282,103.863613,18 Marina Gardens Drive,NaN,This sprawling garden in the city provides mes...,http://www.gardensbythebay.com.sg/en/home.html,Gardens by the Bay provides mesmerising waterf...,"Bay South Outdoor Gardens Daily, 5am - 2am Coo...",76915C78CEAAEB32,20180619175243,103.863613,1.282
106,998,www.yoursingapore.com/en/see-do-singapore/plac...,www.yoursingapore.com/content/dam/desktop/glob...,Immerse yourself in a day of history and cultu...,NaN,"Civic District, Singapore",2015-10-13T14:04:28.278+08:00,1.293,103.852220,150 North Bridge Road,NaN,The Civic District is where Singaporeâ€™s hist...,NaN,The Civic District is where Sing

In [5]:
import folium

# Use LATITUDE/LONGTITUDE or latitude/longitude (from geometry)
lat_col = "LATITUDE" if "LATITUDE" in df.columns else "latitude"
lon_col = "LONGTITUDE" if "LONGTITUDE" in df.columns else "longitude"

# Drop rows with missing coordinates
df_map = df.dropna(subset=[lat_col, lon_col])

# Center map on Singapore
m = folium.Map(location=[1.3521, 103.8198], zoom_start=12)

# Add markers
for _, row in df_map.iterrows():
    popup = row.get("PAGETITLE", row.get("ADDRESS", "—"))
    folium.Marker(
        location=[row[lat_col], row[lon_col]],
        popup=str(popup),
        tooltip=str(popup)
    ).add_to(m)

m.save("map.html")
print("Map saved to map.html — open it in your browser")

Map saved to map.html — open it in your browser


In [6]:
df.columns.tolist()

['OBJECTID_1',
 'URL_PATH',
 'IMAGE_PATH',
 'IMAGE_ALT_TEXT',
 'PHOTOCREDITS',
 'PAGETITLE',
 'LASTMODIFIED',
 'LATITUDE',
 'LONGTITUDE',
 'ADDRESS',
 'POSTALCODE',
 'OVERVIEW',
 'EXTERNAL_LINK',
 'META_DESCRIPTION',
 'OPENING_HOURS',
 'INC_CRC',
 'FMEL_UPD_D',
 'longitude',
 'latitude']

### HDB resale flat prices — dataset size (ingest)

`resale_flat_price` is loaded via **poll-download** (bulk CSV), same as the Airflow DAG. Below: **row count** from CKAN `datastore_search` (cheap, `limit=1`) and **approximate CSV size** from **`Content-Length` on a streaming `GET`** to the presigned S3 URL (no full download). **Do not use `HEAD` on that URL**—S3 presigns are usually **GET-only**, so `HEAD` often returns **403 Forbidden**.

In [9]:
from pathlib import Path
import sys

# Resolve project root so `tourist_attraction_ingest` imports work (run from repo root or this folder).
_cwd = Path.cwd()
for base in (_cwd, _cwd.parent):
    if (base / "tourist_attraction_ingest" / "dag_helpers.py").is_file():
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break
else:
    raise ImportError("Could not find tourist_attraction_ingest/dag_helpers.py — open notebook from hdb-price-estimator or tourist_attraction_ingest.")

from tourist_attraction_ingest import dag_helpers

SOURCES = dag_helpers.SOURCES
session = dag_helpers._http_session()  # retries on 429 for data.gov.sg

cfg = SOURCES["resale_flat_price"]
dataset_id = cfg["dataset_id"]
api_base = cfg["api_base"].rstrip("/")

# --- 1) Row count (CKAN; same resource id, minimal payload) ---
ckan = session.get(
    "https://data.gov.sg/api/action/datastore_search",
    params={"resource_id": dataset_id, "limit": 1},
    timeout=120,
)
ckan.raise_for_status()
body = ckan.json()
res = body.get("result") or {}
total_rows = res.get("total")
sample_cols = list((res.get("records") or [{}])[0].keys()) if res.get("records") else []

# --- 2) Approx. CSV bytes (same poll-download path as Airflow) ---
poll = session.get(f"{api_base}/{dataset_id}/poll-download", timeout=120)
poll.raise_for_status()
pj = poll.json()
code = pj.get("code")
csv_url = (pj.get("data") or {}).get("url")
csv_bytes = None
if csv_url:
    # Presigned S3 URLs are signed for GET. HEAD is a different HTTP method and often returns 403.
    with session.get(csv_url, stream=True, allow_redirects=True, timeout=300) as gr:
        gr.raise_for_status()
        cl = gr.headers.get("Content-Length")
        if cl is not None:
            csv_bytes = int(cl)

def _fmt_bytes(n: int) -> str:
    if n < 1024:
        return f"{n} B"
    val = float(n)
    for unit in ("KiB", "MiB", "GiB"):
        val /= 1024.0
        if val < 1024.0 or unit == "GiB":
            return f"{val:.2f} {unit}"
    return f"{n} B"

print("resale_flat_price (SOURCES key)")
print(f"  dataset_id / resource_id: {dataset_id}")
print(f"  CKAN total rows (datastore_search): {total_rows}")
if sample_cols:
    print(f"  sample columns ({len(sample_cols)}): {sample_cols[:8]}{' ...' if len(sample_cols) > 8 else ''}")
print(f"  poll-download API code: {code}")
if csv_bytes is not None:
    print(f"  CSV Content-Length (GET): {_fmt_bytes(csv_bytes)} ({csv_bytes:,} bytes)")
else:
    print("  CSV size: Content-Length not in GET response headers (chunked/unknown)")
if total_rows and csv_bytes:
    approx = csv_bytes / total_rows
    print(f"  ~ {approx:.1f} bytes/row (CSV; rough)")


resale_flat_price (SOURCES key)
  dataset_id / resource_id: d_8b84c4ee58e3cfc0ece0d773c8ca6abc
  CKAN total rows (datastore_search): 227319
  sample columns (12): ['_id', 'month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm'] ...
  poll-download API code: 0
  CSV Content-Length (GET): 21.43 MiB (22,466,397 bytes)
  ~ 98.8 bytes/row (CSV; rough)
